In [5]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [6]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [7]:
g = load_queries("sql/02g_shared_wav.sql")
sh = run(g, "shared_groups_overview"); display(sh)
wv = run(g, "wav_groups_overview"); display(wv)
print("Uber shared-table rows:", int(sh["n_rows"].sum()), "(expect 15,356,455)")
print("WAV-table rows by company:", wv.groupby("company")["n_rows"].sum().to_dict(), "(expect Uber 15,356,455; Lyft 5,049,211)")

,shared_group,n_rows,n_valid,n_excluded_negative_wait,pct_excluded,wait_p50_min,wait_p90_min,pct_wait_ge10,median_miles
0,"1_shared requested, matched",471930,466787,5143,1.090,7.017,14.200,27.910,5.70
1,"2_shared requested, not matched",365342,361794,3548,0.971,5.700,9.133,6.968,2.33
2,3_not shared,14519183,14406907,112276,0.773,3.700,7.683,4.578,2.82


,company,wav_group,n_rows,n_valid,n_excluded_negative_wait,pct_excluded,wait_p50_min,wait_p90_min,pct_wait_ge10,median_miles,pct_access_a_ride
0,Uber,1_WAV requested,37074,36395,679,1.831,8.067,14.550,31.944,2.80,4.60
1,Uber,"2_WAV vehicle, not requested",1153806,1147070,6736,0.584,3.833,8.033,5.266,2.25,0.08
2,Uber,3_no WAV,14165575,14052023,113552,0.802,3.783,8.017,5.288,2.95,0.08
3,Lyft,1_WAV requested,14076,8822,5254,37.326,5.950,11.500,15.745,2.66,4.52
4,Lyft,"2_WAV vehicle, not requested",649557,638653,10904,1.679,4.017,7.967,4.637,2.96,0.11
5,Lyft,3_no WAV,4385578,4322915,62663,1.429,4.083,8.033,4.756,2.66,0.11


Uber shared-table rows: 15356455 (expect 15,356,455)
WAV-table rows by company: {'Lyft': 5049211, 'Uber': 15356455} (expect Uber 15,356,455; Lyft 5,049,211)


In [9]:
stage3 = load_queries("sql/03_stage3_validation.sql")

print("Available queries:", list(stage3))

display(run(stage3, "stage3_schema_probe"))

Available queries: ['stage3_schema_probe']


,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,trip_miles,DOUBLE,YES,None,None,None


In [10]:
stage3 = load_queries("sql/03_stage3_validation.sql")

hourly_check = run(
    stage3,
    "stage3_hourly_wait_validation"
)

display(hourly_check)

,request_hour,trip_rows,valid_wait_rows,long_wait_rows,median_wait_min,p90_wait_min,long_wait_pct
0,7,932409,911980,99314,4.983,10.267,10.890
1,10,879047,864280,21812,3.350,6.533,2.524


In [11]:
stage3 = load_queries("sql/03_stage3_validation.sql")

daily_hours = run(stage3, "stage3_daily_hour_stability")

display(daily_hours)

print(
    "Days when 7 AM had a higher long-wait rate:",
    int((daily_hours["long_wait_gap_pp"] > 0).sum()),
    "out of",
    len(daily_hours)
)

print(
    "Median daily gap in long-wait rate:",
    round(daily_hours["long_wait_gap_pp"].median(), 3),
    "percentage points"
)

,request_date,valid_rows_7am,valid_rows_10am,long_wait_pct_7am,long_wait_pct_10am,median_wait_7am,median_wait_10am,long_wait_gap_pp,median_wait_gap_min
0,2025-01-01,15413,15743,8.207357,4.490885,5.250000,4.450000,3.716,0.800
1,2025-01-02,27876,23864,8.641842,3.515756,4.766667,3.383333,5.126,1.383
2,2025-01-03,29045,24947,7.660527,3.174730,4.500000,3.366667,4.486,1.133
3,2025-01-04,15523,26666,4.702699,3.285082,3.666667,3.316667,1.418,0.350
4,2025-01-05,14764,27396,5.581143,2.967587,4.033333,3.516667,2.614,0.517
5,2025-01-06,33634,26342,13.183089,2.934477,5.666667,3.333333,10.249,2.333
6,2025-01-07,35871,27476,14.939645,2.667783,5.933333,3.350000,12.272,2.583
7,2025-01-08,37227,27524,13.807183,2.445139,5.600000,3.250000,11.362,2.350
8,2025-01-09,38330,29594,15.345682,2.108536,5.700000,3.200000,13.237,2.500
9,2025-01-10,36263,28121,14.127347,2.144305,5.516667,3.250000,11.983,2.267


Days when 7 AM had a higher long-wait rate: 30 out of 31
Median daily gap in long-wait rate: 6.831 percentage points


In [12]:
import math
from scipy.stats import t

gaps = daily_hours["long_wait_gap_pp"].dropna().tolist()

n = len(gaps)
mean_gap = sum(gaps) / n
sd_gap = (
    sum((x - mean_gap) ** 2 for x in gaps) / (n - 1)
) ** 0.5

se = sd_gap / math.sqrt(n)
critical = t.ppf(0.975, df=n - 1)

print("Days compared:", n)
print("Mean daily long-wait-rate gap:", round(mean_gap, 3), "pp")
print(
    "Approx. 95% CI:",
    round(mean_gap - critical * se, 3),
    "to",
    round(mean_gap + critical * se, 3),
    "pp"
)
print(
    "Days with higher 7 AM long-wait rate:",
    sum(x > 0 for x in gaps),
    "out of",
    n
)

Days compared: 31
Mean daily long-wait-rate gap: 7.018 pp
Approx. 95% CI: 5.055 to 8.981 pp
Days with higher 7 AM long-wait rate: 30 out of 31


In [13]:
print("Available columns:", sh.columns.tolist())

wanted = [
    "shared_group",
    "wait_p90_min",
    "pct_wait_ge10"
]

if all(c in sh.columns for c in wanted):
    display(sh[wanted])
else:
    display(sh)

Available columns: ['shared_group', 'n_rows', 'n_valid', 'n_excluded_negative_wait', 'pct_excluded', 'wait_p50_min', 'wait_p90_min', 'pct_wait_ge10', 'median_miles']


,shared_group,wait_p90_min,pct_wait_ge10
0,"1_shared requested, matched",14.200,27.910
1,"2_shared requested, not matched",9.133,6.968
2,3_not shared,7.683,4.578


In [14]:
g = load_queries("sql/02g_shared_wav.sql")

shared_comparison = run(
    g,
    "shared_vs_not_shared_zone_daypart"
)

display(shared_comparison)

,share_group,n_valid_waits,raw_median_wait_min,raw_p90_wait_min,raw_long_wait_pct,standardized_long_wait_pct,standardized_gap_pp,common_cells,rows_in_common_cells,pct_rows_in_common_cells
0,Not shared,14406907,3.700,7.683,4.578,4.438,13.524,1182,14106890.0,97.92
1,Shared requested,828581,6.217,12.533,18.766,17.963,13.524,1182,826685.0,99.77


In [15]:
from pathlib import Path

keywords = ("hotspot", "stage2d", "design_c", "designc")

for path in sorted(Path("sql").glob("*.sql")):
    queries = load_queries(str(path))
    matches = [
        name for name in queries
        if any(word in name.lower() for word in keywords)
    ]

    if matches or any(word in path.name.lower() for word in keywords):
        print(f"\nFILE: {path}")
        print("Matching queries:", matches)


FILE: sql/02d2_hotspots_zone_hour.sql
Matching queries: ['create_zh_hotspots', 'zh_hotspot_summary', 'zh_hotspot_by_borough', 'zh_hotspot_top_by_rate', 'zh_hotspot_top_non_airport', 'zh_hotspot_top_by_impact', 'zh_hotspot_zone_summary', 'zh_hotspot_by_hour']

FILE: sql/02d3_hotspots_zone_daypart.sql
Matching queries: ['create_zdd_hotspots', 'zdd_hotspot_summary', 'zdd_hotspot_by_slot']

FILE: sql/02d_hotspots.sql
Matching queries: ['create_zhw_hotspots', 'hotspot_summary', 'hotspot_top_by_rate', 'hotspot_top_by_impact', 'hotspot_zone_summary', 'hotspot_by_slot', 'hotspot_by_borough']

FILE: sql/02e_company_like_for_like.sql
Matching queries: ['ce_design_c_crosscheck']


In [16]:
files_to_inspect = [
    ("sql/02d_hotspots.sql",
     ["create_zhw_hotspots", "hotspot_summary"]),

    ("sql/02d2_hotspots_zone_hour.sql",
     ["create_zh_hotspots", "zh_hotspot_summary"]),

    ("sql/02d3_hotspots_zone_daypart.sql",
     ["create_zdd_hotspots", "zdd_hotspot_summary"]),

    ("sql/02e_company_like_for_like.sql",
     ["ce_design_c_crosscheck"]),
]

for file_path, query_names in files_to_inspect:
    queries = load_queries(file_path)

    print(f"\n--- {file_path} ---")

    for name in query_names:
        if name in queries:
            print(f"\nQUERY: {name}")
            print(queries[name])


--- sql/02d_hotspots.sql ---

QUERY: create_zhw_hotspots
-- Adds baselines (window sums over the cell table), the Wilson interval, exclusion bounds and the flags.
-- Wilson 95% interval for a proportion p on n valid waits (z = 1.96, z^2 = 3.8416):
--   center = (p + z^2/(2n)) / (1 + z^2/n);  half = z * sqrt(p(1-p)/n + z^2/(4n^2)) / (1 + z^2/n);  interval = center +/- half.
CREATE OR REPLACE TABLE zhw_hotspots AS
WITH base AS (
    SELECT
        c.*,
        c.n_wait_ge10 * 1.0 / c.n_valid                                          AS p,
        SUM(c.n_valid)     OVER (PARTITION BY c.request_isodow, c.request_hour)  AS slot_n_valid,
        SUM(c.n_wait_ge10) OVER (PARTITION BY c.request_isodow, c.request_hour)  AS slot_n_ge10,
        SUM(c.n_valid)     OVER (PARTITION BY c.PULocationID)                    AS zone_n_valid,
        SUM(c.n_wait_ge10) OVER (PARTITION BY c.PULocationID)                    AS zone_n_ge10
    FROM zhw_cells c
    WHERE c.n_valid > 0
),
w AS (
    SELECT
  

In [17]:
# Show the named queries available in each hotspot SQL file.
hotspot_files = [
    "sql/02d_hotspots.sql",
    "sql/02d2_hotspots_zone_hour.sql",
    "sql/02d3_hotspots_zone_daypart.sql",
]

for path in hotspot_files:
    queries = load_queries(path)
    print(f"\n{path}")
    print(list(queries.keys()))

# Show tables currently available in our DuckDB connection.
print("\nTables currently available:")
display(con.execute("SHOW TABLES").df())


sql/02d_hotspots.sql
['create_zhw_cells', 'create_zhw_hotspots', 'cells_reconcile', 'cell_size_distribution', 'cell_size_quantiles', 'slot_baseline', 'hotspot_summary', 'hotspot_top_by_rate', 'hotspot_top_by_impact', 'hotspot_zone_summary', 'hotspot_by_slot', 'hotspot_by_borough']

sql/02d2_hotspots_zone_hour.sql
['create_zh_cells', 'create_zh_hotspots', 'create_coverage_cells', 'zh_cells_reconcile', 'coverage_tradeoff', 'coverage_tradeoff_by_borough', 'zh_cell_size_distribution', 'zh_slot_baseline', 'zh_hotspot_summary', 'zh_hotspot_by_borough', 'zh_staten_island_zones', 'zh_hotspot_top_by_rate', 'zh_hotspot_top_non_airport', 'zh_hotspot_top_by_impact', 'zh_hotspot_zone_summary', 'zh_hotspot_by_hour', 'zh_vs_fine']

sql/02d3_hotspots_zone_daypart.sql
['create_zdd_cells', 'create_zdd_hotspots', 'create_zdd_zone_severity', 'zdd_cells_reconcile', 'coverage_tradeoff_v2', 'coverage_tradeoff_by_borough_v2', 'zdd_borough_summary', 'zdd_hotspot_summary', 'zdd_hotspot_by_slot', 'zdd_staten_is

,name
0,trips_clean
1,trips_features
2,trips_raw
3,trips_scoped
4,zones


In [18]:
from pathlib import Path

# Find the project decision log without scanning the trip data.
candidates = [
    Path("DECISIONS.md"),
    Path("../DECISIONS.md"),
    Path("reports/DECISIONS.md"),
]

decision_file = next((p for p in candidates if p.exists()), None)

if decision_file is None:
    print("DECISIONS.md not found in the expected locations.")
else:
    print("Reading:", decision_file)
    lines = decision_file.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        if any(term in line.lower() for term in
               ["design c", "133", "hotspot"]):
            start = max(0, i - 2)
            end = min(len(lines), i + 5)
            print("\n".join(lines[start:end]))
            print("---")

Reading: DECISIONS.md

## D11: Pickup-before-request rows (LOCKED, Stage 1)
- **Decision (rule R1):** `is_wait_valid = (wait_seconds >= 0)`. Rows with wait_seconds < 0 are excluded from wait statistics only (wait medians and percentiles, wait hotspots, Stage 4 wait targets). Raw rows are never deleted, and timestamps are never repaired. The same rule applies to Uber and Lyft. Zero waits stay valid (about 0.001% of rows, Stage 1B).
- **Rows affected:** 199,788 of 20,405,666 (0.979%). Uber 120,967 of 15,356,455 (0.788%); Lyft 78,821 of 5,049,211 (1.561%). Rows left for wait statistics: 20,205,878 (Uber 15,235,488; Lyft 4,970,390, by subtraction), before any 1B cleaning.
- **Evidence (confirmed outputs):**
  - Gap size: under 1 min 36,021; 1 to 10 min 98,114; 10 to 60 min 65,649; 60 min or more 4. About 82% are 1 minute or more, so this is not just rounding noise.
  - Uber ordering of request / on_scene / pickup: A normal 15,176,848 (98.830%); B on_scene before request with non-negative w

In [19]:
hotspot = load_queries("sql/02d3_hotspots_zone_daypart.sql")

# Rebuild the daypart hotspot tables in the current DuckDB connection.
con.execute(hotspot["create_zdd_cells"])
con.execute(hotspot["create_zdd_hotspots"])

# Existing summary from the original Stage 2D SQL.
display(run(hotspot, "zdd_hotspot_summary"))

# Calculate impact specifically for the robust hotspot cells.
robust_check = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE hotspot_robust_flag
        ) AS robust_hotspot_cells,

        ROUND(
            100.0 * SUM(n_valid) FILTER (
                WHERE hotspot_robust_flag
            ) / NULLIF(SUM(n_valid), 0),
            2
        ) AS pct_valid_trips_in_robust_hotspots,

        ROUND(
            100.0 * SUM(n_wait_ge10) FILTER (
                WHERE hotspot_robust_flag
            ) / NULLIF(SUM(n_wait_ge10), 0),
            2
        ) AS pct_long_waits_in_robust_hotspots

    FROM zdd_hotspots
""").df()

display(robust_check)

# We need these labels to test the original cells in the second half.
display(con.execute("""
    SELECT DISTINCT daypart, day_type
    FROM zdd_cells
    ORDER BY daypart, day_type
""").df())

,n_cells_with_valid_wait,n_adequate_cells,n_zones_with_adequate_cell,n_hotspot_1_5x,n_hotspot_2x,n_hotspot_3x,n_hotspot_robust_2x,n_time_specific,n_persistent,n_hotspot_in_airport_zones,n_hotspot_ex_airport_baseline,n_added_by_ex_airport_baseline,n_lost_by_ex_airport_baseline,n_zones_with_hotspot
0,2579,2177,241,262,133,47,133,39,94,12,172,39,0,56


,robust_hotspot_cells,pct_valid_trips_in_robust_hotspots,pct_long_waits_in_robust_hotspots
0,133,5.96,22.07


,daypart,day_type
0,1_overnight_0-5,Mon-Fri
1,1_overnight_0-5,Sat-Sun
2,2_morning_6-9,Mon-Fri
3,2_morning_6-9,Sat-Sun
4,3_midday_10-15,Mon-Fri
5,3_midday_10-15,Sat-Sun
6,4_evening_peak_16-19,Mon-Fri
7,4_evening_peak_16-19,Sat-Sun
8,5_late_evening_20-23,Mon-Fri
9,5_late_evening_20-23,Sat-Sun


In [23]:
stage3 = load_queries("sql/03_stage3_validation.sql")

hotspot_persistence = run(
    stage3,
    "stage3_hotspot_second_half_persistence"
)

display(hotspot_persistence.T)

,0
original_hotspot_cells,133.00
cells_with_valid_waits,133.00
cells_still_at_least_2x_slot_rate,128.00
pct_observed_cells_still_2x,96.24
hotspot_valid_waits,625538.00
hotspot_long_waits,125673.00
pct_valid_trips_in_original_hotspots,5.80
pct_long_waits_in_original_hotspots,22.18
hotspot_long_wait_rate_pct,20.09
overall_long_wait_rate_pct,5.25


In [24]:
model_queries = load_queries("sql/04_predictive_model.sql")

con.execute(model_queries["create_stage4_model_sample"])

display(con.execute("""
    SELECT
        split,
        COUNT(*) AS n_rows,
        SUM(target_long_wait) AS long_wait_rows,
        ROUND(100.0 * AVG(target_long_wait), 2) AS long_wait_pct
    FROM stage4_model_sample
    GROUP BY split
    ORDER BY CASE split
        WHEN 'train' THEN 1
        WHEN 'validation' THEN 2
        WHEN 'test' THEN 3
    END
""").df())

,split,n_rows,long_wait_rows,long_wait_pct
0,train,134168,7468.0,5.57
1,validation,20991,1041.0,4.96
2,test,46443,2107.0,4.54


In [25]:
from pathlib import Path

# Confirm split sizes and date boundaries.
display(con.execute("""
    SELECT
        split,
        MIN(request_date) AS first_date,
        MAX(request_date) AS last_date,
        COUNT(*) AS n_rows,
        SUM(target_long_wait) AS long_wait_rows,
        ROUND(100.0 * AVG(target_long_wait), 2) AS long_wait_pct
    FROM stage4_model_sample
    GROUP BY split
    ORDER BY CASE split
        WHEN 'train' THEN 1
        WHEN 'validation' THEN 2
        WHEN 'test' THEN 3
    END
""").df())

# Export the compact modelling sample.
output_path = Path("data/stage4_model_sample.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

sql_path = str(output_path.resolve()).replace("'", "''")

con.execute(
    f"COPY stage4_model_sample TO '{sql_path}' "
    "(FORMAT PARQUET, COMPRESSION ZSTD)"
)

print("File created:", output_path.resolve())
print("File size (MB):", round(output_path.stat().st_size / 1_000_000, 2))

,split,first_date,last_date,n_rows,long_wait_rows,long_wait_pct
0,train,2025-01-01,2025-01-21,134168,7468.0,5.57
1,validation,2025-01-22,2025-01-24,20991,1041.0,4.96
2,test,2025-01-25,2025-01-31,46443,2107.0,4.54


File created: /Users/waheed/Desktop/Project1/data/stage4_model_sample.parquet
File size (MB): 0.3
